# Scale and cost: the same questions over a corpus twelve times larger

Published grep agents lead at small corpora and fall behind ranked retrieval as the corpus grows; one study puts the crossover near 10 million corpus tokens (arXiv:2607.26497, a single study). This notebook runs RI #5's arms on the same MuSiQue questions over two corpora that differ only in size: the 11,656 packaged passages (about 1.5 million tokens) and the 139,416-passage pool of the authors' release (about 16 million tokens), which holds the 11,656 under the same identifiers. A question's supporting passages are the same at both sizes; only the distractors grow.

On these 120 questions the published crossover does not appear at 16 million tokens; the larger corpus hurts dense search more than grep. `C5` holds at F1 0.553 and 0.556 as the corpus grows twelvefold, while `C0` falls from 0.517 to 0.469, so `C5`'s lead grows from +0.036 (95% interval -0.033 to +0.105) to +0.087 (+0.018 to +0.155), and on the 40 four-hop questions to +0.142 (+0.020 to +0.275). Grep pays for the larger corpus in calls: `C5`'s calls per question rise from 14.4 to 16.1 and its budget exhaustion from 12% to 23%, because the median bridge grep now returns 88 passages instead of 13. `C7`, which bounds grep with a dense search, is the most accurate arm at 11,656 passages (F1 0.559) and second at 139,416 (0.513), and the most expensive, at 107,000 and 140,000 tokens per question. All three stay under 45 calls a question; `C7` is about a quarter of the way to the 590,000-token standard. One model, one run per arm: the trend is the finding, not the third decimal.

On `gemini-3.5-flash-lite`, the default model, the direction is the same. `C5` and `C0` tie at 11,656 passages, and at 139,416 `C5` leads on all 75 four-hop questions by +0.115 F1 (+0.030 to +0.205), while over the 120 ladder questions the lead stays inside the noise. `C7` is the least accurate arm at 139,416. The section on the default model has the table.


Three arms: `C0` (dense `search` only), `C5` (`grep` and `open` only), and `C7`, which searches first (top 20, short previews) and may then `grep` only inside what its searches returned, the bounded-workspace pattern of RISE (arXiv:2606.06880). The runs use the first 40 questions of each hop count in the shared 300 (120 questions), on the on-prem `Qwen3_6-35B-A3B` with thinking off and a 30-call budget. Each arm ran once, with no repeats, so only large effects are readable.

The reference runs are read from the cache. Rebuilding them is not part of the hands-on notebook time: about 9,000 calls. An optional cell runs 10 questions live on your model. The saved outputs below were executed with `ADVANCED_RAG_MODEL_FAMILY=onprem`, so the arm tables in the next section show the `Qwen3_6-35B-A3B` reference trajectories this paragraph cites. On the default family, those cells read the `gemini-3.5-flash-lite` trajectories instead, which the section on the default model compares.

MuSiQue (Trivedi, Balasubramanian, Khot, and Sabharwal, TACL 2022) is licensed CC BY 4.0.

In [1]:
import json
import os
from pathlib import Path

from agent_arms import dense_search, ladder_questions, live_questions, musique_corpus, run_arm
from agent_report import public_row, summarise
from aieng.advanced_rag import spend
from aieng.advanced_rag.corpora.musique import load_musique, stratified_test_set
from aieng.advanced_rag.models import choose_models
from grep_probe import reachability


os.environ.setdefault("HF_HUB_OFFLINE", "1")

# The live cell calls your model. False reads the reference trajectories.
RUN_EXPENSIVE = False
MODELS = choose_models(show=True)

_, all_questions = load_musique()
SHARED = stratified_test_set(all_questions)
LADDER = ladder_questions(SHARED)
SIZES = (11_656, 139_416)
corpora = {size: musique_corpus(size) for size in SIZES}
for size, corpus in corpora.items():
    tokens = corpus.characters // 4
    print(
        f"{size:>7,} passages  {corpus.characters:>12,} characters  about {tokens / 1e6:.1f}M tokens  ({tokens / 10e6:.2f}x the 10M crossover)"
    )
print(f"{len(LADDER)} questions: " + ", ".join(f"{h}-hop {sum(q.hops == h for q in LADDER)}" for h in (2, 3, 4)))

family onprem
  generation  Qwen3_6-35B-A3B         $0 (Vector cluster)
  judging     Qwen3_6-35B-A3B         $0 (Vector cluster)
  extraction  Qwen3_6-35B-A3B         $0 (Vector cluster)
  embeddings  BAAI/bge-small-en-v1.5  local
key budget: not shown in the committed output
 11,656 passages     5,869,862 characters  about 1.5M tokens  (0.15x the 10M crossover)
139,416 passages    64,176,325 characters  about 16.0M tokens  (1.60x the 10M crossover)
120 questions: 2-hop 40, 3-hop 40, 4-hop 40


## What grep sees as the corpus grows

Before any model call: the same model-free probe as notebook 02, at both sizes, read from `outputs/musique_grep_probe.json` (`python grep_probe.py` rebuilds it in a few minutes). A literal grep for the right span still finds the right passage in the larger pool; what changes is how many other passages match with it, which is what the agent has to read past.

In [2]:
# The probe scans the pool once per span; grep_probe.py wrote its result.
if RUN_EXPENSIVE:
    probes = {size: reachability(LADDER, corpus) for size, corpus in corpora.items()}
    probes = {size: {**p, "by_hop": {str(k): v for k, v in p["by_hop"].items()}} for size, p in probes.items()}
else:
    saved = json.loads(Path("outputs/musique_grep_probe.json").read_text())
    probes = {size: saved[f"ladder-{size}"] for size in SIZES}
print(f"{'passages':>9} {'hops':>5} {'first hop':>10} {'bridge':>7} {'chain':>6}")
for size, probe in probes.items():
    for hops, row in probe["by_hop"].items():
        print(f"{size:>9,} {hops:>5} {row['first_hop']:>10.3f} {row['bridge']:>7.3f} {row['chain']:>6.3f}")
print()
for size, probe in probes.items():
    h = probe["hit_sizes"]
    print(
        f"{size:>9,}: passages per bridge grep, median {h['bridge_median']}, 90th percentile {h['bridge_p90']}; first-hop span median {h['first_hop_median']}, 90th percentile {h['first_hop_p90']}"
    )

 passages  hops  first hop  bridge  chain
   11,656     2      0.875   0.950  0.825
   11,656     3      0.900   0.762  0.625
   11,656     4      0.725   0.808  0.450
  139,416     2      0.875   0.950  0.825
  139,416     3      0.900   0.762  0.625
  139,416     4      0.725   0.808  0.450

   11,656: passages per bridge grep, median 13.0, 90th percentile 87; first-hop span median 2.0, 90th percentile 76
  139,416: passages per bridge grep, median 88.0, 90th percentile 848; first-hop span median 2.5, 90th percentile 687


## The arms at both sizes

All-supporting recall is over the passages each run read in full. `zero-hit greps` is the share of `grep` calls that matched nothing, and `narrowed` the share that matched more than 200 passages and came back with a request to narrow the pattern instead of previews.

In [3]:
gold = {q.question_id: q.supporting_ids for q in LADDER}
summaries = {}
raw = {}
for size, corpus in corpora.items():
    for arm in ("C0", "C5", "C7"):
        result = run_arm(arm, LADDER, corpus=corpus, label=f"ladder-{size}", models=MODELS, live=False, show=False)
        rows = [public_row(r) for r in result.value["runs"]]
        raw[(size, arm)] = result.value["runs"]
        summaries[(size, arm)] = summarise(rows, gold)
print(f"model: {result.value['model']}, reference trajectories")

model: Qwen3_6-35B-A3B, reference trajectories


In [4]:
def grep_shares(runs):
    """Return the share of grep calls with no hit and with too many to show."""
    greps = [e for r in runs for e in r.get("trace", []) if e["name"] == "grep"]
    if not greps:
        return None, None
    zero = sum(e["total_hits"] == 0 for e in greps) / len(greps)
    narrowed = sum((e["total_hits"] or 0) > 200 for e in greps) / len(greps)
    return zero, narrowed


print(
    f"{'passages':>9} {'arm':<4} {'F1':>6} {'EM':>6} {'recall':>7} {'calls/q':>8} {'tokens/q':>9} {'exhaust':>8} {'zero-hit':>9} {'narrowed':>9}"
)
for (size, arm), summary in summaries.items():
    s = summary["all"]
    zero, narrowed = grep_shares(raw[(size, arm)])
    zero_text = f"{zero:.2f}" if zero is not None else "-"
    narrowed_text = f"{narrowed:.2f}" if narrowed is not None else "-"
    print(
        f"{size:>9,} {arm:<4} {s['f1']:>6.3f} {s['exact_match']:>6.3f} {s['all_supporting_recall_read']:>7.3f} "
        f"{s['calls_per_question']:>8.1f} {s['tokens_per_question']:>9,.0f} {s['exhaustion_rate']:>8.2f} {zero_text:>9} {narrowed_text:>9}"
    )

 passages arm      F1     EM  recall  calls/q  tokens/q  exhaust  zero-hit  narrowed
   11,656 C0    0.517  0.392   0.458      9.3    88,553     0.12         -         -
   11,656 C5    0.553  0.467   0.250     14.4    56,578     0.12      0.61      0.00
   11,656 C7    0.559  0.467   0.175     11.7   106,588     0.09      0.59      0.00
  139,416 C0    0.469  0.383   0.308     10.1   100,059     0.19         -         -
  139,416 C5    0.556  0.458   0.200     16.1    70,808     0.23      0.60      0.05
  139,416 C7    0.513  0.425   0.167     13.2   140,117     0.17      0.55      0.00


In [5]:
print(
    f"{'arm':<4} {'hops':>5} "
    + " ".join(f"{'F1 @' + format(size, ','):>12}" for size in SIZES)
    + "   "
    + " ".join(f"{'recall @' + format(size, ','):>16}" for size in SIZES)
)
for arm in ("C0", "C5", "C7"):
    for group in ("2hop", "3hop", "4hop"):
        f1 = " ".join(f"{summaries[(size, arm)][group]['f1']:>12.3f}" for size in SIZES)
        recall = " ".join(f"{summaries[(size, arm)][group]['all_supporting_recall_read']:>16.3f}" for size in SIZES)
        print(f"{arm:<4} {group:>5} {f1}   {recall}")

arm   hops   F1 @11,656  F1 @139,416     recall @11,656  recall @139,416
C0    2hop        0.682        0.611              0.775            0.525
C0    3hop        0.356        0.374              0.400            0.300
C0    4hop        0.513        0.422              0.200            0.100
C5    2hop        0.729        0.711              0.525            0.375
C5    3hop        0.406        0.393              0.150            0.175
C5    4hop        0.522        0.564              0.075            0.050
C7    2hop        0.720        0.629              0.400            0.400
C7    3hop        0.405        0.375              0.100            0.100
C7    4hop        0.553        0.536              0.025            0.000


## Cost per question against corpus size

Tokens per question are what a paid model bills. The standard RI #4 used to drop Microsoft GraphRAG was 45 calls and 590,000 tokens per question.

In [6]:
for arm in ("C0", "C5", "C7"):
    small, large = (summaries[(size, arm)]["all"] for size in SIZES)
    growth = (
        large["tokens_per_question"] / small["tokens_per_question"] if small["tokens_per_question"] else float("nan")
    )
    print(
        f"{arm:<4} tokens/q {small['tokens_per_question']:>8,.0f} -> {large['tokens_per_question']:>8,.0f} ({growth:.2f}x)   "
        f"calls/q {small['calls_per_question']:.1f} -> {large['calls_per_question']:.1f}   "
        f"F1 {small['f1']:.3f} -> {large['f1']:.3f}"
    )

C0   tokens/q   88,553 ->  100,059 (1.13x)   calls/q 9.3 -> 10.1   F1 0.517 -> 0.469
C5   tokens/q   56,578 ->   70,808 (1.25x)   calls/q 14.4 -> 16.1   F1 0.553 -> 0.556
C7   tokens/q  106,588 ->  140,117 (1.31x)   calls/q 11.7 -> 13.2   F1 0.559 -> 0.513


## The ladder on the default model

The same 120 questions, both corpus sizes, and the three arms were rerun on `gemini-3.5-flash-lite`, the default model for participants, and judged by the shared judge `gemini-3.8-flash`. `outputs/musique_scale_ladder_gemini.json` holds the runs, labelled a paid-model rerun in its `produced_by` block. The question: does `C5`'s lead over `C0` return when dense top-5 search faces twelve times the distractors? A separate run, `outputs/musique_agent_arms_4hop_139416_gemini.json`, covers all 75 four-hop questions at 139,416 passages.

`C7` first: the most accurate arm on `Qwen3_6-35B-A3B` at 11,656 passages ties `C5` on Gemini there (F1 0.555) and is the least accurate at 139,416 (0.483), at three to four times the tokens of `C0` or `C5`. It sees every supporting passage in a preview for 72% of questions at 11,656 passages and opens all of them for 14%. For `C7` to win on this model, the agent would have to open what its top-20 previews point to; the README's `C7-top5-full` follow-up tests a top-5 search with full previews on the 40 four-hop ladder questions.

`C5`'s lead returns on the four-hop questions. On all 75 at 139,416 passages `C5` leads by +0.115 F1 (95% interval +0.030 to +0.205) and +0.133 judged, where at 11,656 the two tie (-0.026). Over the 120 ladder questions the lead stays inside the noise: -0.009 F1 at 11,656 and +0.015 at 139,416 (-0.059 to +0.090), and on the 40 ladder four-hop questions it moves from +0.000 to +0.095. On Qwen the same ladder shows `C5` ahead at 139,416 by +0.087 over the 120. `C0` still reads every supporting passage for 42% of questions at 139,416, against 25% for `C5`. These are single runs with no repeats, on one answering model and one question set.

In [7]:
gemini = json.loads(Path("outputs/musique_scale_ladder_gemini.json").read_text(encoding="utf-8"))
reference = json.loads(Path("outputs/musique_scale_ladder.json").read_text(encoding="utf-8"))
four_hop = json.loads(Path("outputs/musique_agent_arms_4hop_139416_gemini.json").read_text(encoding="utf-8"))
block = gemini["produced_by"]
print(f"model: {block['model']}, {block['label']}, judged by {block['judge']}")
print(
    f"{'passages':>9} {'arm':<4} {'F1 Qwen':>8} {'F1':>6} {'EM':>6} {'judged':>7} {'4-hop F1':>9} "
    f"{'read':>6} {'seen':>6} {'calls/q':>8} {'tokens/q':>9} {'exhaust':>8}"
)
for size, entry in gemini["sizes"].items():
    for arm, value in entry["arms"].items():
        s = value["summary"]["all"]
        qwen = reference["sizes"][size]["arms"][arm]["summary"]["all"]["f1"]
        print(
            f"{int(size):>9,} {arm:<4} {qwen:>8.3f} {s['f1']:>6.3f} {s['exact_match']:>6.3f} {value['judged']['all']:>7.3f} "
            f"{value['summary']['4hop']['f1']:>9.3f} {s['all_supporting_recall_read']:>6.3f} "
            f"{s['all_supporting_recall_seen']:>6.3f} {s['calls_per_question']:>8.1f} {s['tokens_per_question']:>9,.0f} "
            f"{s['exhaustion_rate']:>8.2f}"
        )
print()
for size, entry in gemini["sizes"].items():
    for name, value in entry["comparisons"].items():
        print(
            f"{int(size):>9,} {name}: {value['mean_difference']:+.3f} over {value['questions']} questions, "
            f"95% interval [{value['low']:+.3f}, {value['high']:+.3f}]"
        )
value = four_hop["comparisons"]["C5_minus_C0_f1_4hop"]
judged = {arm: four_hop["arms"][arm]["judged"]["4hop"] for arm in ("C0", "C5")}
print(
    f"  139,416 all 75 four-hop questions: {value['mean_difference']:+.3f} F1, "
    f"95% interval [{value['low']:+.3f}, {value['high']:+.3f}]; judged C0 {judged['C0']:.3f}, C5 {judged['C5']:.3f}"
)

model: gemini-3.5-flash-lite, paid-model rerun, judged by gemini-3.8-flash
 passages arm   F1 Qwen     F1     EM  judged  4-hop F1   read   seen  calls/q  tokens/q  exhaust
   11,656 C0      0.517  0.564  0.475   0.583     0.541  0.517  0.517      6.5    22,551     0.00
   11,656 C5      0.553  0.555  0.467   0.583     0.542  0.317  0.742     11.2    34,964     0.01
   11,656 C7      0.559  0.555  0.467   0.567     0.529  0.142  0.717     10.1    87,552     0.01
  139,416 C0      0.469  0.534  0.442   0.542     0.524  0.417  0.417      7.4    31,355     0.01
  139,416 C5      0.556  0.549  0.467   0.558     0.619  0.250  0.550     12.6    41,496     0.04
  139,416 C7      0.513  0.483  0.400   0.475     0.430  0.092  0.517     11.6   129,697     0.07

   11,656 C5_minus_C0_f1_4hop: +0.000 over 40 questions, 95% interval [-0.110, +0.108]
   11,656 C5_minus_C0_f1_all: -0.009 over 120 questions, 95% interval [-0.076, +0.060]
  139,416 C5_minus_C0_f1_4hop: +0.095 over 40 questions, 95% int

## Run 10 questions on your model

Optional, and only with `RUN_EXPENSIVE = True`: `C5` and `C7` on 10 questions over the 139,416-passage pool. Embedding the pool for `C7` takes several minutes on a CPU the first time and is cached after.

In [8]:
if RUN_EXPENSIVE:
    subset = live_questions(LADDER, "claude-opus-5-5")  # 10 questions on any tier
    pool = corpora[139_416]
    for arm in ("C5", "C7"):
        value = run_arm(
            arm,
            subset,
            corpus=pool,
            label="ladder-live-139416",
            models=MODELS,
            live=True,
            search=lambda: dense_search(pool),
        ).value
        rows = [public_row(r) for r in value["runs"]]
        s = summarise(rows, gold)["all"]
        print(
            f"{arm} on {value['model']}: F1 {s['f1']:.3f}, recall {s['all_supporting_recall_read']:.3f}, {s['calls_per_question']:.1f} calls/q, {s['tokens_per_question']:,.0f} tokens/q"
        )
else:
    print("Set RUN_EXPENSIVE = True to run C5 and C7 on your model over the pool.")

Set RUN_EXPENSIVE = True to run C5 and C7 on your model over the pool.


In [9]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
